# LoMa EDisGo-Workshop 13.2.2025

Contents:

Import Packages

In [ ]:
%load_ext jupyter_black

In [ ]:
import os
import requests
import sys

import matplotlib.pyplot as plt
import networkx as nx
import pandas as pd

from copy import deepcopy
from numpy.random import default_rng
from pathlib import Path

from edisgo import EDisGo
from edisgo.io.db import engine
from edisgo.tools.logger import setup_logger

In [ ]:
# Nur wegen der Übersicht. Normalerweise nicht zu empfehlen
import warnings

warnings.filterwarnings("ignore")

## Load grid Data from ding0

Currently, synthetic grid data generated with the python project ding0 is the only supported data source for distribution grid data. ding0 provides the grid topology data in the form of csv files, with separate files for buses, lines, loads, generators, etc. You can retrieve ding0 data from Zenodo (make sure you choose latest data) or check out the Ding0 documentation on how to generate grids yourself. A ding0 example grid can be viewed here. It is possible to provide your own grid data if it is in the same format as the ding0 grid data.

This example works with any ding0 grid data. If you don't have grid data yet, you can execute the following to download the example grid data mentioned above.

The ding0 grid you want to use in your analysis is specified through the input parameter 'ding0_grid' of the EDisGo class. The following assumes you want to use the ding0 example grid downloaded above. To use a different ding0 grid, just change the path below.

In [ ]:
ding0_grid = os.path.join(
    os.path.expanduser("~"), ".ding0", "run_2025-02-04-09-34-25", "35725"
)
edisgo = EDisGo(ding0_grid=ding0_grid, legacy_ding0_grids=False)

## Plot grid topology (MV)

In [ ]:
sizes_dict = {
    "BranchTee": 10000,
    "GeneratorFluctuating": 100000,
    "Generator": 100000,
    "Load": 100000,
    "LVStation": 50000,
    "MVStation": 120000,
    "Storage": 100000,
    "DisconnectingPoint": 75000,
    "else": 200000,
}

sizes_dict = {k: v / 10 for k, v in sizes_dict.items()}

In [ ]:
edisgo.plot_mv_grid_topology(technologies=True, sizes_dict=sizes_dict)

red: nodes with substation secondary side
light blue: nodes distribution substations's primary side
green: nodes with fluctuating generators
black: nodes with conventional generators
grey: disconnecting points
dark blue: branch trees

Geladenes Netz muss auf den aktuellen Stand gestzt werden --> reinforce

worst case analysis ... 

In [ ]:
edisgo.set_time_series_worst_case_analysis()

In [ ]:
edisgo.reinforce()

In [ ]:
edisgo.topology.generators_df.head()

### TODO: Hier etwas die eDisGo Struktur zeigen und ein paar Statistiken wie folgende

In [ ]:
edisgo.topology.generators_df[["p_nom", "type"]].groupby("type").sum()

## Adapt network (Husum)

### Basic components addition and removal

To see how a loaded network can be adapted later on, we add a solar plant to a random bus. Then we remove it again. 

Components can also be added according to their geolocation with the function integrate_component_based_on_geolocation().

In [ ]:
edisgo.topology.generators_df

Add a generator with the function add_component or add_generator.

In [ ]:
rng = default_rng(1)
rnd_bus = rng.choice(edisgo.topology.buses_df.index, size=1)[0]
generator_type = "solar"

new_generator = edisgo.add_component(
    comp_type="generator", p_nom=0.01, bus=rnd_bus, generator_type=generator_type
)

Mit Generator zeigen, mit Last nachmachen

In [ ]:
edisgo.topology.generators_df

In [ ]:
edisgo.remove_component(comp_type="generator", comp_name=new_generator)

In [ ]:
edisgo.topology.generators_df

### Add flexible components to grid (Heat pumps)

Add heat pumps with the function import_heat_pumps()

Engine der eigentlichen Datenbank muss verwendet werden

In [ ]:
scenario = "eGon2035"

In [ ]:
conf_path = (
    Path.home()
    / "Documents"
    / "data"
    / "egon-data-hetzner"
    / "egon-data.configuration.yaml"
)

db_engine = engine(path=conf_path, ssh=True)

In [ ]:
edisgo_orig = deepcopy(edisgo)

In [ ]:
# Retry if running into "Connection reset by peer" error
edisgo = deepcopy(edisgo_orig)

edisgo.import_generators(generator_scenario=scenario)
edisgo.import_home_batteries(scenario=scenario, engine=db_engine)
edisgo.import_heat_pumps(scenario=scenario, engine=db_engine)

In [ ]:
# This takes too long for the workshop, but needs to be mentioned
# edisgo_obj.import_dsm(scenario=scenario, engine=db_engine)
# edisgo_obj.import_electromobility(
#     data_source="oedb", scenario=scenario, engine=db_engine
# )

In [ ]:
edisgo.topology.generators_df[["p_nom", "type"]].groupby("type").sum()

In [ ]:
# TODO Moritz: Show statistics before and after import
edisgo.topology.generators_df.head()

edisgo.import_heat_pumps(scenario = scenario, engine = toep_engine())

Batteries: import_home_batteries()

Demand Side Management: import_dsm()

Charging parks and stations for EV: import_electromobility() --> See electromobility example

## Create timeseries for all intergrated components

Create timeseries for the four worst cases MV load case, LV load case, MV feed-in case, LV feed-in case with the function  set_time_series_worst_case_analysis():

In [ ]:
edisgo.set_time_series_worst_case_analysis()

In [ ]:
# TODO: Erklärung zu der Unterscheidung der vier timesteps
edisgo.timeseries.timeindex_worst_cases

In [ ]:
edisgo.timeseries.loads_active_power.loc[
    edisgo.timeseries.timeindex_worst_cases["load_case_mv"]
]

## Main features

Execute a power flow analysis to determine line overloads and voltage deviations for the MV load case timeseries with the function analyze():

In [ ]:
edisgo.analyze(timesteps=edisgo.timeseries.timeindex_worst_cases["load_case_mv"])

In [ ]:
edisgo.plot_mv_line_loading(
    node_color="voltage_deviation",
    timestep=edisgo.timeseries.timeindex_worst_cases["load_case_mv"],
)

In [ ]:
edisgo.histogram_voltage(binwidth=0.005)

In [ ]:
edisgo.histogram_relative_line_load(binwidth=0.1, voltage_level="mv")

Reinfoce the grid with the function reinforce(): (Use mode 'mv' for shorter runtime)

In [ ]:
edisgo.reinforce(mode="mv")

In [ ]:
edisgo.plot_mv_line_loading(
    node_color="voltage_deviation",
    timestep=edisgo.timeseries.timeindex_worst_cases["load_case_mv"],
)

In [ ]:
edisgo.histogram_voltage(binwidth=0.005)

In [ ]:
edisgo.histogram_relative_line_load(binwidth=0.1, voltage_level="mv")

## Results

Display the resulting equipment changes and their corresponding costs with the results class:

In [ ]:
edisgo.results.equipment_changes.head()

### TODO: Die initialen Netzausbaumaßnahmen am Anfang müssen hier abgezogen werden

In [ ]:
edisgo.results.grid_expansion_costs.head()

### TODO: Visualisierung der Ergebnisse

### TODO: Alternative Wege für Zeitreihen, wie von der DB

* show time resolution